# 01 Cloudless DET Feedback GA Search v3 Clean Final

기존 notebook의 중복 함수, `row1_*` 변수 꼬임, 미정의 변수 `a`, GT/generated 출력 혼선을 제거한 최종 단일 notebook입니다.

## 실행 목차

0. Clean setup  
1. Worker preflight  
2. Category 5/6 debug row 선택  
3. Baseline single-row GA  
4. 실제 테스트 row compact table  
5. GT vs Generated 병렬 비교  
6. mutation / feedback evidence  
7. actual prompt 원문  
8. manual feedback rule 설계  
9. patched genome/source_file 생성  
10. before/after genome diff  
11. 같은 row 재실행  
12. patched run compact table  
13. patched run GT vs Generated  
14. GT / before / after 3열 비교  
15. before/after evaluation table  
16. patched prompt 원문 확인  
17. category sweep  
18. full 280 × 10 generations  
19. final plots  

In [ ]:
# ============================================================
# Cell 0. Clean setup: server preset, paths, runners, viewers
# ============================================================

import os, sys, json, shlex, html, difflib, subprocess
from pathlib import Path
from datetime import datetime

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML

import re
import textwrap

pd.set_option("display.max_columns", 240)
pd.set_option("display.width", 260)
pd.set_option("display.max_colwidth", 260)

# -------------------------------
# 0.1 Server preset
# -------------------------------
SERVER_PRESET = os.environ.get("SERVER_PRESET", "a100")  # "a100" or "a6000"
MODEL_KEY_VALUE = os.environ.get("MODEL_KEY", "qwen25_coder_14b")

if SERVER_PRESET == "a100":
    os.environ.setdefault("JOILANG_BASE_DIR", "/root/llm/JOILang-Server")
    os.environ.setdefault("JOI_V15_LOCAL_MODEL_BASE_DIR", "/root/llm/local_models")
    os.environ.setdefault("JOI_V15_LOCAL_MODEL_NAME", f"/root/llm/local_models/{MODEL_KEY_VALUE}")
    os.environ.setdefault("JOI_V15_LOCAL_LOAD_IN_4BIT", "false")
elif SERVER_PRESET == "a6000":
    os.environ.setdefault("JOILANG_BASE_DIR", "/home/mgjeong/Desktop/llm/JOILang-Server")
    os.environ.setdefault("JOI_V15_LOCAL_MODEL_BASE_DIR", "/home/mgjeong/Desktop/llm/local_models")
    os.environ.setdefault("JOI_V15_LOCAL_MODEL_NAME", f"/home/mgjeong/Desktop/llm/local_models/{MODEL_KEY_VALUE}")
    os.environ.setdefault("JOI_V15_LOCAL_LOAD_IN_4BIT", "true")
else:
    raise ValueError(f"Unknown SERVER_PRESET={SERVER_PRESET!r}")

os.environ.setdefault("MODEL_KEY", MODEL_KEY_VALUE)
os.environ.setdefault("JOI_V15_LOCAL_DEVICE", "cuda:0")
os.environ.setdefault("JOI_V15_LOCAL_FILES_ONLY", "true")
os.environ.setdefault("JOI_V15_LOCAL_DTYPE", "bf16")
os.environ.setdefault("JOI_V15_LOCAL_TRUST_REMOTE_CODE", "true")


'true'

In [3]:
# -------------------------------
# 0.2 Paths
# -------------------------------
BASE_DIR = Path(os.environ["JOILANG_BASE_DIR"]).expanduser().resolve()
VERSION_ROOT = BASE_DIR / "gpt_mg" / "version0_15_update20260413"
GA_SCRIPT = VERSION_ROOT / "scripts" / "run_ga_search.py"
RUN_BENCHMARK = VERSION_ROOT / "scripts" / "run_benchmark.py"

DATASET = BASE_DIR / "datasets" / "JOICommands-280.csv"
SERVICE_SCHEMA = BASE_DIR / "datasets" / "service_list_ver2.0.1.json"
DEFAULT_GENOME = VERSION_ROOT / "genomes" / "example_genome.json"

PYTHON = os.environ.get("JOI_V15_PYTHON", sys.executable)
WORKER_PYTHON = os.environ.get("JOI_V15_WORKER_PYTHON", PYTHON)
MODEL_KEY = os.environ["MODEL_KEY"]
DEVICE = os.environ["JOI_V15_LOCAL_DEVICE"]

LOCAL_MODELS_BASE = Path(os.environ["JOI_V15_LOCAL_MODEL_BASE_DIR"]).expanduser().resolve()
LOCAL_MODEL_DIR = Path(os.environ["JOI_V15_LOCAL_MODEL_NAME"]).expanduser().resolve()

RUN_TAG = os.environ.get("RUN_TAG", datetime.now().strftime("%Y%m%d_%H%M%S"))
NOTEBOOK_RUN_ROOT = BASE_DIR / "artifacts" / "notebook_ga_runs" / RUN_TAG
NOTEBOOK_RUN_ROOT.mkdir(parents=True, exist_ok=True)

LLM_EXTRA_JSON = NOTEBOOK_RUN_ROOT / "llm_extra_worker_preflight.json"
LLM_EXTRA_JSON.write_text(json.dumps({
    "local_model_name": str(LOCAL_MODEL_DIR),
    "local_files_only": os.environ.get("JOI_V15_LOCAL_FILES_ONLY", "true").lower() == "true",
    "local_device": DEVICE,
    "local_dtype": os.environ.get("JOI_V15_LOCAL_DTYPE", "bf16"),
    "local_load_in_4bit": os.environ.get("JOI_V15_LOCAL_LOAD_IN_4BIT", "false").lower() == "true",
    "local_trust_remote_code": os.environ.get("JOI_V15_LOCAL_TRUST_REMOTE_CODE", "true").lower() == "true",
}, ensure_ascii=False, indent=2), encoding="utf-8")

ENV = os.environ.copy()
ENV.update({
    "JOI_V15_PYTHON": PYTHON,
    "JOI_V15_WORKER_PYTHON": WORKER_PYTHON,
    "JOI_V15_LOCAL_MODEL_BASE_DIR": str(LOCAL_MODELS_BASE),
    "JOI_V15_LOCAL_MODEL_NAME": str(LOCAL_MODEL_DIR),
    "JOI_V15_LOCAL_FILES_ONLY": os.environ.get("JOI_V15_LOCAL_FILES_ONLY", "true"),
    "JOI_V15_LOCAL_DEVICE": DEVICE,
    "JOI_V15_LOCAL_DTYPE": os.environ.get("JOI_V15_LOCAL_DTYPE", "bf16"),
    "JOI_V15_LOCAL_LOAD_IN_4BIT": os.environ.get("JOI_V15_LOCAL_LOAD_IN_4BIT", "false"),
    "JOI_V15_LOCAL_TRUST_REMOTE_CODE": os.environ.get("JOI_V15_LOCAL_TRUST_REMOTE_CODE", "true"),
    "TRANSFORMERS_VERBOSITY": "error",
    "HF_HUB_DISABLE_PROGRESS_BARS": "1",
    "TOKENIZERS_PARALLELISM": "false",
    "PYTHONFAULTHANDLER": "1",
})

def _print_path(name, p):
    p = Path(p)
    print(f"{name}: {p} exists={p.exists()}")

print("SERVER_PRESET:", SERVER_PRESET)
_print_path("BASE_DIR", BASE_DIR)
_print_path("VERSION_ROOT", VERSION_ROOT)
_print_path("GA_SCRIPT", GA_SCRIPT)
_print_path("RUN_BENCHMARK", RUN_BENCHMARK)
_print_path("DATASET", DATASET)
_print_path("SERVICE_SCHEMA", SERVICE_SCHEMA)
_print_path("DEFAULT_GENOME", DEFAULT_GENOME)
_print_path("LOCAL_MODEL_DIR", LOCAL_MODEL_DIR)
print("DEVICE:", DEVICE)
print("LOAD_IN_4BIT:", ENV["JOI_V15_LOCAL_LOAD_IN_4BIT"])
print("NOTEBOOK_RUN_ROOT:", NOTEBOOK_RUN_ROOT)

assert BASE_DIR.exists(), BASE_DIR
assert VERSION_ROOT.exists(), VERSION_ROOT
assert GA_SCRIPT.exists(), GA_SCRIPT
assert RUN_BENCHMARK.exists(), RUN_BENCHMARK
assert DATASET.exists(), DATASET
assert SERVICE_SCHEMA.exists(), SERVICE_SCHEMA
assert DEFAULT_GENOME.exists(), DEFAULT_GENOME
assert LOCAL_MODEL_DIR.exists(), LOCAL_MODEL_DIR
assert str(LOCAL_MODEL_DIR) != "/root/llm/JOILang-Server/local_models/qwen25_coder_14b", "Bad A100 local model path."


SERVER_PRESET: a100
BASE_DIR: /root/llm/JOILang-Server exists=True
VERSION_ROOT: /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413 exists=True
GA_SCRIPT: /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/scripts/run_ga_search.py exists=True
RUN_BENCHMARK: /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/scripts/run_benchmark.py exists=True
DATASET: /root/llm/JOILang-Server/datasets/JOICommands-280.csv exists=True
SERVICE_SCHEMA: /root/llm/JOILang-Server/datasets/service_list_ver2.0.1.json exists=True
DEFAULT_GENOME: /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/genomes/example_genome.json exists=True
LOCAL_MODEL_DIR: /root/llm/local_models/qwen25_coder_14b exists=True
DEVICE: cuda:0
LOAD_IN_4BIT: false
NOTEBOOK_RUN_ROOT: /root/llm/JOILang-Server/artifacts/notebook_ga_runs/20260623_180959


In [ ]:
# -------------------------------
# 0.3 Basic utilities
# -------------------------------
def ts():
    return datetime.now().strftime("%Y%m%d_%H%M%S")

def run_cmd(cmd, *, cwd=BASE_DIR, env=ENV, log_path=None, check=True):
    cmd = [str(x) for x in cmd]
    print("\n[CMD]")
    print(" ".join(shlex.quote(x) for x in cmd))
    if log_path:
        log_path = Path(log_path)
        log_path.parent.mkdir(parents=True, exist_ok=True)
        print("[LOG]", log_path)
    proc = subprocess.Popen(cmd, cwd=str(cwd), env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    lines = []
    with (open(log_path, "w", encoding="utf-8") if log_path else open(os.devnull, "w", encoding="utf-8")) as lf:
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end="")
            lines.append(line)
            if log_path:
                lf.write(line)
    rc = proc.wait()
    output = "".join(lines)
    if check and rc != 0:
        raise RuntimeError(f"command failed rc={rc}: {' '.join(cmd)}")
    return rc, output

def load_json(path):
    path = Path(path)
    return json.loads(path.read_text(encoding="utf-8")) if path.exists() else {}

def dump_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")
    return path

def safe_json_loads(x, default=None):
    if x is None:
        return default
    if isinstance(x, (list, dict)):
        return x
    try:
        if isinstance(x, float) and pd.isna(x):
            return default
    except Exception:
        pass
    try:
        return json.loads(str(x))
    except Exception:
        return default

def read_csv_if_exists(path):
    path = Path(path)
    return pd.read_csv(path) if path.exists() else pd.DataFrame()

# -------------------------------
# 0.4 GA runner
# -------------------------------
def ga_common_args(genome_json=None, mutation_mode="cloudless_decompiler"):
    genome_json = Path(genome_json or DEFAULT_GENOME).expanduser().resolve()
    return [
        PYTHON, "-u", str(GA_SCRIPT),
        "--profile", "version0_15",
        "--genome-json", str(genome_json),
        "--dataset", str(DATASET),
        "--service-schema", str(SERVICE_SCHEMA),
        "--model-key", MODEL_KEY,
        "--llm-mode", "worker",
        "--candidate-k", "1",
        "--repair-attempts", "0",
        "--det-profile", "strict",
        "--selection-mode", "redesign",
        "--fitness-mode", "phase_aware",
        "--mutation-mode", mutation_mode,
        "--category-balance-mode", "guard",
        "--token-penalty-mode", "hybrid",
        "--stop-controller-mode", "active",
        "--reasoning-mutation-mode", "auto",
        "--intent-hint-mode", "auto",
        "--feedback-guided-mutation",
        "--enable-compression-mutation",
        "--enable-prompt-decompiler",
        "--enable-rendered-prompt-dedupe",
        "--enable-pareto-archive",
        "--enable-group-specialist-archives",
        "--full-run",
        "--force",
        "--progress", "verbose",
        "--retries", "0",
        "--target-detpass", "90",
    ]

def run_ga(label, scope_args, tuning_args=None, extra_args=None, output_root=None, genome_json=None, mutation_mode="cloudless_decompiler", check=True):
    output_root = Path(output_root or (NOTEBOOK_RUN_ROOT / label)).resolve()
    output_root.mkdir(parents=True, exist_ok=True)
    log_path = output_root / f"{label}.log"
    cmd = ga_common_args(genome_json=genome_json, mutation_mode=mutation_mode)
    cmd += list(scope_args)
    cmd += list(tuning_args or [])
    cmd += ["--output-root", str(output_root)]
    cmd += list(extra_args or [])
    run_cmd(cmd, log_path=log_path, check=check)
    return output_root

def run_worker_preflight(check=False):
    log_path = NOTEBOOK_RUN_ROOT / f"worker_preflight_{ts()}.log"
    cmd = [
        PYTHON, str(RUN_BENCHMARK),
        "--suite", "paper_local5",
        "--model-key", MODEL_KEY,
        "--prompt-render-mode", "blocks",
        "--llm-extra-json", str(LLM_EXTRA_JSON),
        "--preflight-only",
        "--print-worker-info",
        "--strict-availability",
    ]
    return run_cmd(cmd, log_path=log_path, check=check)

# -------------------------------
# 0.5 Dataset / candidate extraction
# -------------------------------
def dataset_df():
    return pd.read_csv(DATASET)

def load_dataset_row(row_no):
    ds = dataset_df()
    if "row_no" in ds.columns:
        hit = ds.loc[ds["row_no"].astype(str) == str(row_no)]
        if not hit.empty:
            return hit.iloc[0].to_dict()
    return ds.iloc[int(row_no) - 1].to_dict()

def _find_col(keys, exact, contains=None):
    lower = {k.lower(): k for k in keys}
    for item in exact:
        if item.lower() in lower:
            return lower[item.lower()]
    if contains:
        for k in keys:
            lk = k.lower()
            if any(t.lower() in lk for t in contains):
                return k
    return None

def extract_gt_fields(row_no):
    row = load_dataset_row(row_no)
    keys = list(row.keys())
    eng_col = _find_col(keys, ["command_eng", "command_en", "english", "eng", "sentence_en", "nl_en"])
    kor_col = _find_col(keys, ["command_kor", "command_ko", "korean", "kor", "sentence_ko", "nl_ko"])
    code_col = _find_col(keys, ["code", "gt_code", "joi_code", "joilang", "gt_joilang", "answer", "target", "target_code", "ground_truth_code"], contains=["gt", "code", "joilang"])
    cron_col = _find_col(keys, ["cron", "gt_cron"])
    period_col = _find_col(keys, ["period", "gt_period"])
    cat_col = _find_col(keys, ["category", "cat"])
    return {
        "row_no": row_no,
        "category": row.get(cat_col, "") if cat_col else "",
        "eng": row.get(eng_col, "") if eng_col else "",
        "kor": row.get(kor_col, "") if kor_col else "",
        "gt_cron": row.get(cron_col, "") if cron_col else "",
        "gt_period": row.get(period_col, "") if period_col else "",
        "gt_code": row.get(code_col, "") if code_col else "",
        "raw": row,
    }

def collect_candidate_tables(run_dir):
    run_dir = Path(run_dir)
    dfs = []
    for p in sorted(run_dir.rglob("*.csv")):
        try:
            head = pd.read_csv(p, nrows=1)
        except Exception:
            continue
        cols = set(head.columns)
        if {"genome_id", "candidates"}.issubset(cols) or {"genome_id", "prompt_log_paths"}.issubset(cols):
            df = pd.read_csv(p)
            df["source_file"] = str(p)
            dfs.append(df)
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

def collect_tested_candidate_rows(run_dir, row_no=None, phase="validation"):
    df = collect_candidate_tables(run_dir)
    if df.empty:
        return df
    if row_no is not None and "row_no" in df.columns:
        df = df[df["row_no"].astype(str) == str(row_no)].copy()
    if phase != "all" and "source_file" in df.columns:
        key = "ga_validation" if phase == "validation" else "ga_quick"
        phase_df = df[df["source_file"].astype(str).str.contains(key, na=False)].copy()
        if not phase_df.empty:
            df = phase_df

    generated, strategies, errors = [], [], []
    for _, r in df.iterrows():
        cand = safe_json_loads(r.get("candidates", ""), default=[])
        generated.append(cand[0] if isinstance(cand, list) and cand else (cand if isinstance(cand, str) else ""))
        meta = safe_json_loads(r.get("candidate_metadata", ""), default=[])
        if isinstance(meta, list) and meta:
            strategies.append(meta[0].get("strategy", ""))
            errors.append(meta[0].get("error", ""))
        elif isinstance(meta, dict):
            strategies.append(meta.get("strategy", ""))
            errors.append(meta.get("error", ""))
        else:
            strategies.append("")
            errors.append("")
    df["generated_code"] = generated
    df["candidate_strategy"] = strategies
    df["candidate_error"] = errors
    dedupe_cols = [c for c in ["row_no", "genome_id", "generated_code", "source_file"] if c in df.columns]
    if dedupe_cols:
        df = df.drop_duplicates(subset=dedupe_cols)
    return df.reset_index(drop=True)

def detect_worker_crash(run_dir):
    df = collect_candidate_tables(run_dir)
    if df.empty:
        return False
    text = df.astype(str).to_string()
    return "worker_crash" in text or "Repo id must be in the form" in text

# -------------------------------
# 0.6 Viewers
# -------------------------------

def _normalize_code_text(text):
    if text is None:
        return ""

    text = str(text)

    # 문자열 안에 literal "\\n"이 들어온 경우 일부 복원
    text = text.replace("\\r\\n", "\n").replace("\\n", "\n").replace("\\t", "    ")

    # 들여쓰기 정리
    text = textwrap.dedent(text).strip()

    # 과도한 빈 줄은 최대 1줄로 압축
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text

def _format_joilang_for_display(raw):
    """
    GT / generated code가 아래 형태 중 무엇이든 보기 좋게 변환한다.

    1) {"name": "", "cron": "", "period": 0, "script": "..."}
    2) {"name": "", "cron": "", "period": 0, "code": "..."}
    3) raw JOILang script
    4) invalid JSON string
    """
    if raw is None:
        return ""

    if not isinstance(raw, str):
        try:
            raw = json.dumps(raw, ensure_ascii=False)
        except Exception:
            raw = str(raw)

    raw = raw.strip()
    if not raw:
        return ""

    try:
        obj = json.loads(raw)

        if isinstance(obj, dict):
            script_key = None
            for k in ["script", "code", "joilang", "target_code", "gt_code"]:
                if k in obj:
                    script_key = k
                    break

            if script_key:
                meta = dict(obj)
                script = meta.pop(script_key)

                meta_text = json.dumps(meta, ensure_ascii=False, indent=2)
                script_text = _normalize_code_text(script)

                return (
                    "[JSON meta]\n"
                    f"{meta_text}\n\n"
                    f"[{script_key}]\n"
                    f"{script_text}"
                )

            return json.dumps(obj, ensure_ascii=False, indent=2)

        if isinstance(obj, list):
            return json.dumps(obj, ensure_ascii=False, indent=2)

    except Exception:
        pass

    return _normalize_code_text(raw)

def _code_pre_html(title, content):
    formatted = _format_joilang_for_display(content)

    return f"""
    <div>
      <div style="
        font-weight:700;
        background:#f7f7f7;
        padding:6px;
        border:1px solid #eee;
        border-bottom:none;
      ">{html.escape(str(title))}</div>
      <pre style="
        margin:0;
        white-space:pre;
        overflow:auto;
        max-height:520px;
        min-height:180px;
        tab-size:4;
        font-family:Consolas, 'Courier New', monospace;
        font-size:13px;
        line-height:1.45;
        background:#fbfbfb;
        padding:10px;
        border:1px solid #eee;
      ">{html.escape(formatted)}</pre>
    </div>
    """
def summarize_ga_run(run_dir):
    run_dir = Path(run_dir)
    s = load_json(run_dir / "ga_summary.json")
    b = load_json(run_dir / "best_genome.json")
    display(pd.DataFrame([{
        "run_dir": str(run_dir),
        "best_DETPass": s.get("best_DETPass") or s.get("accepted_best_DETPass"),
        "best_avg_DET": s.get("best_avg_DET") or s.get("accepted_best_avg_DET"),
        "best_prompt_tokens": s.get("best_avg_prompt_tokens") or s.get("accepted_best_avg_prompt_tokens"),
        "best_genome_id": b.get("id") or b.get("genome_id"),
        "stop_reason": s.get("stop_reason"),
    }]))
    for name in ["ga_summary.json", "best_genome.json", "ga_block_diffs.jsonl", "ga_generation_progress.csv", "pareto_rows.csv"]:
        p = run_dir / name
        print(f"{name}: {p.exists()}  {p}")
    return s, b

def show_tested_rows_only(run_dir, row_no=None, phase="validation", max_rows=30):
    df = collect_tested_candidate_rows(run_dir, row_no=row_no, phase=phase)
    if df.empty:
        print("No tested candidate rows found.")
        return df
    preferred = ["category", "row_no", "generation", "genome_id", "base_genome_id", "candidate_strategy", "prompt_render_mode", "generation_error_type", "generation_error_count", "generation_prompt_tokens_total", "generation_completion_tokens_total", "generated_code", "candidate_error", "source_file"]
    auto = [c for c in df.columns if any(k in c.lower() for k in ["det", "pass", "failure", "score", "exact", "sim"])]
    cols = [c for c in preferred + auto if c in df.columns]
    print(f"run_dir={run_dir}")
    print(f"row_no={row_no}, phase={phase}, rows={len(df)}")
    display(df[cols].head(max_rows))
    return df

def show_gt_vs_generated(run_dir, row_no, phase="validation", show_all_genomes=True, max_items=20):
    gt = extract_gt_fields(row_no)
    df = collect_tested_candidate_rows(run_dir, row_no=row_no, phase=phase)
    if df.empty:
        print("No candidate rows found.")
        return df
    if not show_all_genomes:
        non_empty = df[df["generated_code"].astype(str).str.strip().ne("")]
        df = non_empty.head(1) if not non_empty.empty else df.head(1)
    else:
        df = df.head(max_items)

    cards = []
    for _, r in df.iterrows():
        genome_id = r.get("genome_id", "")
        strategy = r.get("candidate_strategy", "")
        gen_err = r.get("generation_error_type", "")
        generated = r.get("generated_code", "")
        cand_err = r.get("candidate_error", "")
        det_lines = []
        for c in df.columns:
            if any(k in c.lower() for k in ["det", "pass", "failure", "exact", "sim", "score"]):
                val = r.get(c, "")
                if str(val) not in ["", "nan", "None"]:
                    det_lines.append(f"<b>{html.escape(c)}</b>: {html.escape(str(val))}")
        cards.append(f"""
        <div style="border:1px solid #ddd; border-radius:8px; padding:12px; margin:12px 0;">
          <div style="font-weight:700;">Genome: {html.escape(str(genome_id))}</div>
          <div style="font-size:13px; color:#444; margin-bottom:8px;">strategy={html.escape(str(strategy))} | generation_error_type={html.escape(str(gen_err))}</div>
          <div style="display:grid; grid-template-columns:1fr 1fr; gap:12px;">
            {_code_pre_html("GT code", gt["gt_code"])}
            {_code_pre_html("Generated code", generated)}
          </div>
          <div style="margin-top:8px; font-size:13px;">{"<br>".join(det_lines[:16])}</div>
          {f'<div style="margin-top:8px; color:#a00;"><b>candidate_error:</b> {html.escape(str(cand_err))}</div>' if cand_err else ''}
        </div>
        """)
    header = f"""
    <div style="border:2px solid #444; border-radius:8px; padding:12px; margin-bottom:12px;">
      <div><b>Run:</b> {html.escape(str(run_dir))}</div>
      <div><b>Row:</b> {row_no}</div>
      <div><b>Category:</b> {html.escape(str(gt["category"]))}</div>
      <div><b>ENG:</b> {html.escape(str(gt["eng"]))}</div>
      <div><b>KOR:</b> {html.escape(str(gt["kor"]))}</div>
      <div><b>GT schedule:</b> cron={html.escape(str(gt["gt_cron"]))}, period={html.escape(str(gt["gt_period"]))}</div>
    </div>
    """
    display(HTML(header + "\n".join(cards)))
    return df

def show_before_after_generated_parallel(before_run, after_run, row_no, phase="validation"):
    gt = extract_gt_fields(row_no)
    bdf = collect_tested_candidate_rows(before_run, row_no=row_no, phase=phase)
    adf = collect_tested_candidate_rows(after_run, row_no=row_no, phase=phase)
    if bdf.empty or adf.empty:
        print("before or after candidate rows missing")
        return None
    def pick(df):
        non_empty = df[df["generated_code"].astype(str).str.strip().ne("")]
        return non_empty.iloc[0] if not non_empty.empty else df.iloc[0]
    b = pick(bdf)
    a = pick(adf)
    text = f"""
    <div style="border:2px solid #333; border-radius:8px; padding:12px; margin-bottom:12px;">
    <div><b>Row:</b> {row_no}</div>
    <div><b>Category:</b> {html.escape(str(gt["category"]))}</div>
    <div><b>ENG:</b> {html.escape(str(gt["eng"]))}</div>
    <div><b>KOR:</b> {html.escape(str(gt["kor"]))}</div>
    </div>

    <div style="display:grid; grid-template-columns:minmax(0, 1fr) minmax(0, 1fr) minmax(0, 1fr); gap:12px;">
    {_code_pre_html("GT", gt["gt_code"])}

    <div>
        <div style="font-size:12px; margin:4px 0;">
            <b>Before</b><br>
            genome={html.escape(str(b.get("genome_id", "")))}<br>
            strategy={html.escape(str(b.get("candidate_strategy", "")))}<br>
            error={html.escape(str(b.get("generation_error_type", "")))}
        </div>
        {_code_pre_html("Before generated", b.get("generated_code", ""))}
    </div>

    <div>
        <div style="font-size:12px; margin:4px 0;">
            <b>After</b><br>
            genome={html.escape(str(a.get("genome_id", "")))}<br>
            strategy={html.escape(str(a.get("candidate_strategy", "")))}<br>
            error={html.escape(str(a.get("generation_error_type", "")))}
            </div>
            {_code_pre_html("After generated", a.get("generated_code", ""))}
        </div>
    </div>
    """
    display(HTML(text))
    return bdf, adf

def compare_two_runs(before_run, after_run, row_no=None, phase="validation"):
    b = collect_tested_candidate_rows(before_run, row_no=row_no, phase=phase)
    a = collect_tested_candidate_rows(after_run, row_no=row_no, phase=phase)
    if b.empty or a.empty:
        print("candidate table missing")
        return pd.DataFrame()
    b["side"] = "before"
    a["side"] = "after"
    combined = pd.concat([b, a], ignore_index=True, sort=False)
    cols = [c for c in ["side", "category", "row_no", "generation", "genome_id", "candidate_strategy", "generation_error_type", "generation_prompt_tokens_total", "generation_completion_tokens_total", "generated_code", "candidate_error", "source_file"] if c in combined.columns]
    display(combined[cols].head(100))
    return combined

# -------------------------------
# 0.7 Prompt mutation helpers
# -------------------------------
def inspect_prompt_log_from_candidates(run_dir, row_no=None, phase="validation", row_index=0, max_chars=8000):
    df = collect_tested_candidate_rows(run_dir, row_no=row_no, phase=phase)
    if df.empty or "prompt_log_paths" not in df.columns:
        print("No prompt log paths found.")
        return None
    paths = safe_json_loads(df.iloc[row_index]["prompt_log_paths"], default=[])
    if not paths:
        print("empty prompt_log_paths")
        return None
    p = Path(paths[0])
    print("prompt log:", p, p.exists())
    if not p.exists():
        return None
    log = json.loads(p.read_text(encoding="utf-8"))
    req = log.get("request", {})
    system = req.get("system", "")
    user = req.get("user", "")
    print("\n=== SYSTEM HEAD ===")
    print(system[:max_chars])
    print("\n=== USER HEAD ===")
    print(user[:max_chars])
    print("\n=== USER TAIL ===")
    print(user[-max_chars:])
    return log

def show_prompt_mutation_evidence(run_dir):
    run_dir = Path(run_dir)
    s = load_json(run_dir / "ga_summary.json")
    b = load_json(run_dir / "best_genome.json")
    display(pd.DataFrame([{
        "best_DETPass": s.get("best_DETPass") or s.get("accepted_best_DETPass"),
        "best_avg_DET": s.get("best_avg_DET") or s.get("accepted_best_avg_DET"),
        "best_prompt_tokens": s.get("best_avg_prompt_tokens") or s.get("accepted_best_avg_prompt_tokens"),
        "stop_reason": s.get("stop_reason"),
    }]))
    print("best genome id:", b.get("id") or b.get("genome_id"))
    print("blocks:", b.get("blocks"))
    print("params:", json.dumps(b.get("params", {}), ensure_ascii=False, indent=2)[:3000])
    print("block_params:", json.dumps(b.get("block_params", {}), ensure_ascii=False, indent=2)[:6000])
    diff_path = run_dir / "ga_block_diffs.jsonl"
    print("\nDiff:", diff_path, diff_path.exists())
    if diff_path.exists():
        rows = [json.loads(x) for x in diff_path.read_text(encoding="utf-8").splitlines() if x.strip()]
        if rows:
            display(pd.DataFrame(rows).head(200))

def resolve_block_source_file(genome, block_id):
    params = (genome.get("block_params") or {}).get(block_id, {}) or {}
    if params.get("source_file"):
        p = VERSION_ROOT / "blocks" / params["source_file"]
        if p.exists():
            return p
    for pattern in [f"{block_id}_*.txt", f"{block_id}*.txt", f"{block_id}_*.md", f"{block_id}*.md"]:
        files = sorted((VERSION_ROOT / "blocks").glob(pattern))
        if files:
            return files[0]
    raise FileNotFoundError(f"Cannot resolve block source for block_id={block_id}")

def make_manual_patched_genome(base_genome_path, *, rules, target_block_id="02", patch_name=None, force_source_file_patch=True):
    base_genome_path = Path(base_genome_path)
    genome = load_json(base_genome_path)
    if not genome:
        raise ValueError(f"empty genome: {base_genome_path}")
    patch_name = patch_name or f"manual_patch_{target_block_id}_{ts()}"
    old_id = genome.get("id", "genome")
    genome["id"] = f"{old_id}__{patch_name}"
    genome.setdefault("block_params", {})
    genome["block_params"].setdefault(target_block_id, {})
    old_rules = list(genome["block_params"][target_block_id].get("micro_rules") or [])
    merged = old_rules[:]
    for rule in rules:
        if rule not in merged:
            merged.append(rule)
    genome["block_params"][target_block_id]["micro_rules"] = merged[-12:]
    if force_source_file_patch:
        source_path = resolve_block_source_file(genome, target_block_id)
        source_text = source_path.read_text(encoding="utf-8")
        marker = "NOTEBOOK MANUAL FOCUS RULES"
        rule_text = "\n".join(f"- {rule}" for rule in rules)
        patched_text = source_text.rstrip() + f"\n\n{marker}\n{rule_text}\n"
        rel = f"generated/{source_path.stem}__{patch_name}{source_path.suffix or '.txt'}"
        out_block = VERSION_ROOT / "blocks" / rel
        out_block.parent.mkdir(parents=True, exist_ok=True)
        out_block.write_text(patched_text, encoding="utf-8")
        genome["block_params"][target_block_id]["source_file"] = rel
        genome["_notebook_manual_block_patch"] = {"target_block_id": target_block_id, "source_path": str(source_path), "patched_block_path": str(out_block), "marker": marker}
    genome["_notebook_manual_patch"] = {"created_at": datetime.now().isoformat(timespec="seconds"), "target_block_id": target_block_id, "rules": rules, "base_genome_path": str(base_genome_path)}
    out = NOTEBOOK_RUN_ROOT / "patched_genomes" / f"{patch_name}.json"
    dump_json(out, genome)
    return out

def diff_json_objects(before_path, after_path, max_lines=220):
    before = json.dumps(load_json(before_path), ensure_ascii=False, indent=2, sort_keys=True).splitlines()
    after = json.dumps(load_json(after_path), ensure_ascii=False, indent=2, sort_keys=True).splitlines()
    diff = list(difflib.unified_diff(before, after, fromfile=str(before_path), tofile=str(after_path), lineterm=""))
    print("\n".join(diff[:max_lines]))
    if len(diff) > max_lines:
        print(f"\n... truncated: {len(diff) - max_lines} more lines")
    return diff

# -------------------------------
# 0.8 Plot helpers
# -------------------------------
def generation_history(run_dir):
    run_dir = Path(run_dir)
    progress = run_dir / "ga_generation_progress.csv"
    if progress.exists():
        df = pd.read_csv(progress)
        df["run_dir"] = str(run_dir)
        return df
    s = load_json(run_dir / "ga_summary.json")
    hist = s.get("best_history") or s.get("generation_history") or []
    if not hist:
        return pd.DataFrame()
    df = pd.DataFrame(hist)
    df["run_dir"] = str(run_dir)
    return df

def collect_run_table(run_dirs):
    rows = []
    for rd in map(Path, run_dirs):
        s = load_json(rd / "ga_summary.json")
        b = load_json(rd / "best_genome.json")
        rows.append({"run_dir": str(rd), "label": rd.name, "best_DETPass": s.get("best_DETPass") or s.get("accepted_best_DETPass"), "best_avg_DET": s.get("best_avg_DET") or s.get("accepted_best_avg_DET"), "best_prompt_tokens": s.get("best_avg_prompt_tokens") or s.get("accepted_best_avg_prompt_tokens"), "stop_reason": s.get("stop_reason"), "best_genome_id": b.get("id") or b.get("genome_id")})
    return pd.DataFrame(rows)

def plot_generation_history(run_dirs):
    dfs = [generation_history(rd) for rd in run_dirs]
    dfs = [df for df in dfs if not df.empty]
    if not dfs:
        print("No generation history found.")
        return pd.DataFrame()
    hist = pd.concat(dfs, ignore_index=True)
    display(hist.head())
    gen_col = "generation" if "generation" in hist.columns else hist.columns[0]
    for y_col, title in [
        (next((c for c in ["validation_det_pass_rate", "train_det_pass_rate", "best_DETPass", "DETPass"] if c in hist.columns), None), "GA DETPass by generation"),
        (next((c for c in ["validation_avg_det_score", "avg_det_score", "best_avg_DET", "avg_DET"] if c in hist.columns), None), "GA average DET by generation"),
        (next((c for c in ["avg_prompt_tokens", "best_avg_prompt_tokens", "prompt_tokens"] if c in hist.columns), None), "Prompt-token trend by generation"),
    ]:
        if y_col:
            plt.figure(figsize=(8, 4))
            for rd, g in hist.groupby("run_dir"):
                plt.plot(g[gen_col], g[y_col], marker="o", label=Path(rd).name)
            plt.xlabel("Generation")
            plt.ylabel(y_col)
            plt.title(title)
            plt.legend()
            plt.grid(True, alpha=0.3)
            plt.show()
    return hist

print("\n[SETUP COMPLETE]")


[SETUP COMPLETE]


## Cell 1. Local worker path preflight

In [5]:
print("LOCAL_MODEL_DIR:", LOCAL_MODEL_DIR)
print("exists:", LOCAL_MODEL_DIR.exists())
assert LOCAL_MODEL_DIR.exists(), LOCAL_MODEL_DIR

print("\nModel directory sample:")
for p in sorted(LOCAL_MODEL_DIR.iterdir())[:40]:
    print(" -", p.name)

RUN_WORKER_PREFLIGHT = True
if RUN_WORKER_PREFLIGHT:
    rc, out = run_worker_preflight(check=False)
    print("preflight rc:", rc)
    if "Repo id must be in the form" in out or "worker_crash" in out:
        raise RuntimeError("Worker preflight indicates bad model path or worker crash.")

LOCAL_MODEL_DIR: /root/llm/local_models/qwen25_coder_14b
exists: True

Model directory sample:
 - .cache
 - .gitattributes
 - LICENSE
 - README.md
 - config.json
 - generation_config.json
 - merges.txt
 - model-00001-of-00006.safetensors
 - model-00002-of-00006.safetensors
 - model-00003-of-00006.safetensors
 - model-00004-of-00006.safetensors
 - model-00005-of-00006.safetensors
 - model-00006-of-00006.safetensors
 - model.safetensors.index.json
 - tokenizer.json
 - tokenizer_config.json
 - vocab.json

[CMD]
/root/llm/je/bin/python /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/scripts/run_benchmark.py --suite paper_local5 --model-key qwen25_coder_14b --prompt-render-mode blocks --llm-extra-json /root/llm/JOILang-Server/artifacts/notebook_ga_runs/20260623_180959/llm_extra_worker_preflight.json --preflight-only --print-worker-info --strict-availability
[LOG] /root/llm/JOILang-Server/artifacts/notebook_ga_runs/20260623_180959/worker_preflight_20260623_181005.log
Preflight:
 -

## Cell 2. Debug row 선택: category 5 또는 6

In [6]:
DEBUG_ROW_CANDIDATES = pd.DataFrame([
    {"row_no": 137, "category": 5, "reason": "temporal/sequential action; delay/order prompt debugging"},
    {"row_no": 170, "category": 6, "reason": "compound condition; condition/action mapping debugging"},
    {"row_no": 141, "category": 5, "reason": "temporal sequence variant"},
    {"row_no": 176, "category": 6, "reason": "compound condition/action variant"},
])
display(DEBUG_ROW_CANDIDATES)

ROW_NO = 170  # 필요하면 170으로 변경
gt = extract_gt_fields(ROW_NO)

display(pd.DataFrame([{
    "row_no": ROW_NO,
    "category": gt["category"],
    "eng": gt["eng"],
    "kor": gt["kor"],
    "gt_cron": gt["gt_cron"],
    "gt_period": gt["gt_period"],
    "gt_code": gt["gt_code"],
}]))

,row_no,category,reason
0,137,5,temporal/sequential action; delay/order prompt debugging
1,170,6,compound condition; condition/action mapping debugging
2,141,5,temporal sequence variant
3,176,6,compound condition/action variant


,row_no,category,eng,kor,gt_cron,gt_period,gt_code
0,170,6,"If the bathroom humidity is 85% or higher and the bathroom door is closed, turn on the light and open the door.","욕실 습도가 85% 이상이고 욕실 문이 닫혀 있으면, 조명을 켜고 문을 열어줘.",,,"{""name"": """", ""cron"": """", ""period"": 0, ""script"": ""\nif ((#Bathroom #HumiditySensor).Humidity >= 85 and (#Bathroom #Door).DoorState == \""closed\"") {\n\n (#Bathroom #Light).On()\n\n (#Bathroom #Door).Open()\n\n}""}"


## Cell 3. Baseline single-row GA 실행

In [12]:
ROW_TUNING = [
    "--population", "4",
    "--gens", "3",
    "--min-generations", "2",
    "--max-generations", "3",
    "--sample-size", "1",
    "--validation-size", "1",
    "--cheap-eval-limit", "1",
    "--plateau-window", "1",
    "--disruptive-max-attempts", "1",
    "--timeout-sec", "2400",
]

baseline_run = run_ga(
    label=f"baseline_row{ROW_NO:03d}_cat{gt['category']}_g3_{ts()}",
    scope_args=["--start-row", str(ROW_NO), "--end-row", str(ROW_NO)],
    tuning_args=ROW_TUNING,
)

summary_before, best_before = summarize_ga_run(baseline_run)

if detect_worker_crash(baseline_run):
    raise RuntimeError("worker_crash detected. Fix LOCAL_MODEL_DIR / local model env before prompt experiments.")


[CMD]
/root/llm/je/bin/python -u /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/scripts/run_ga_search.py --profile version0_15 --genome-json /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/genomes/example_genome.json --dataset /root/llm/JOILang-Server/datasets/JOICommands-280.csv --service-schema /root/llm/JOILang-Server/datasets/service_list_ver2.0.1.json --model-key qwen25_coder_14b --llm-mode worker --candidate-k 1 --repair-attempts 0 --det-profile strict --selection-mode redesign --fitness-mode phase_aware --mutation-mode cloudless_decompiler --category-balance-mode guard --token-penalty-mode hybrid --stop-controller-mode active --reasoning-mutation-mode auto --intent-hint-mode auto --feedback-guided-mutation --enable-compression-mutation --enable-prompt-decompiler --enable-rendered-prompt-dedupe --enable-pareto-archive --enable-group-specialist-archives --full-run --force --progress verbose --retries 0 --target-detpass 90 --start-row 170 --end-row 170 --pop

,run_dir,best_DETPass,best_avg_DET,best_prompt_tokens,best_genome_id,stop_reason
0,/root/llm/JOILang-Server/artifacts/notebook_ga_runs/20260623_180959/baseline_row170_cat6_g3_20260623_181654,100.0,100.0,None,gen-442856f6-c8cd-a33f-a37d-5ce08b9fdc88,max generations reached


ga_summary.json: True  /root/llm/JOILang-Server/artifacts/notebook_ga_runs/20260623_180959/baseline_row170_cat6_g3_20260623_181654/ga_summary.json
best_genome.json: True  /root/llm/JOILang-Server/artifacts/notebook_ga_runs/20260623_180959/baseline_row170_cat6_g3_20260623_181654/best_genome.json
ga_block_diffs.jsonl: True  /root/llm/JOILang-Server/artifacts/notebook_ga_runs/20260623_180959/baseline_row170_cat6_g3_20260623_181654/ga_block_diffs.jsonl
ga_generation_progress.csv: True  /root/llm/JOILang-Server/artifacts/notebook_ga_runs/20260623_180959/baseline_row170_cat6_g3_20260623_181654/ga_generation_progress.csv
pareto_rows.csv: False  /root/llm/JOILang-Server/artifacts/notebook_ga_runs/20260623_180959/baseline_row170_cat6_g3_20260623_181654/pareto_rows.csv


## Cell 4. 실제 테스트된 row만 compact table로 확인

In [13]:
baseline_candidates = show_tested_rows_only(
    baseline_run,
    row_no=ROW_NO,
    phase="validation",
    max_rows=30,
)

run_dir=/root/llm/JOILang-Server/artifacts/notebook_ga_runs/20260623_180959/baseline_row170_cat6_g3_20260623_181654
row_no=170, phase=validation, rows=12


,category,row_no,genome_id,candidate_strategy,prompt_render_mode,generation_error_type,generation_error_count,generation_prompt_tokens_total,generation_completion_tokens_total,generated_code,candidate_error,source_file,service_list_retrieval_scores
0,6,170,gen-016ea579-8fd6-f65f-78f5-12caf55da9da,compact_json,blocks,NaN,0,12814,96,"{""name"":""BathroomHumidityAndDoorCheck"",""cron"":"""",""period"":0,""code"":""if ((#Bathroom #HumiditySensor).humiditysensor_humidity >= 85 && (#Bathroom #Door).door_doorstate == \""closed\"") {\n (#Bathroom #Light).switch_on();\n (#Bathroom #Door).door_open();\n}""}",,/root/llm/JOILang-Server/artifacts/notebook_ga_runs/20260623_180959/baseline_row170_cat6_g3_20260623_181654/candidates/candidates_ga_validation_Qwen_Qwen2.5-Coder-14B-Instruct_gen-016ea579-8fd6-f65f-78f5-12caf55da9da_500845.csv,[]
1,6,170,gen-1b17001a-25af-cc4a-c634-cc693b2989e7,compact_json,blocks,NaN,0,12939,90,"{""name"":""BathroomHumidityAndDoorCheck"",""cron"":"""",""period"":0,""code"":""if ((#Bathroom #HumiditySensor).humiditysensor_humidity >= 85 && (#Bathroom #Door).door_doorstate == \""closed\"") { (#Bathroom #Light).switch_on(); (#Bathroom #Door).door_open(); }""}",,/root/llm/JOILang-Server/artifacts/notebook_ga_runs/20260623_180959/baseline_row170_cat6_g3_20260623_181654/candidates/candidates_ga_validation_Qwen_Qwen2.5-Coder-14B-Instruct_gen-1b17001a-25af-cc4a-c634-cc693b2989e7_500184.csv,[]
2,6,170,gen-378892e9-ecc3-87ab-8b45-85023a0286cc,minimal,blocks,NaN,0,22410,94,"{""name"":""BathroomHumidityAndDoorCheck"",""cron"":"""",""period"":0,""code"":""wait until (((#Bathroom #HumiditySensor).humiditysensor_humidity >= 85) and ((#Bathroom #Door).door_doorstate == \""closed\""))\n(#Bathroom #Light).switch_on()\n(#Bathroom #Door).door_open()""}",,/root/llm/JOILang-Server/artifacts/notebook_ga_runs/20260623_180959/baseline_row170_cat6_g3_20260623_181654/candidates/candidates_ga_validation_Qwen_Qwen2.5-Coder-14B-Instruct_gen-378892e9-ecc3-87ab-8b45-85023a0286cc_507846.csv,[]
3,6,170,gen-3b1ea01c-673b-58d1-52da-f6d4d9796538,compact_json,blocks,NaN,0,21138,92,"{""name"":""BathroomHumidityCheck"",""cron"":"""",""period"":0,""code"":""if ((#Bathroom #HumiditySensor).humiditysensor_humidity >= 85 and (#Bathroom #Door).door_doorstate == \""closed\"") {\n (#Bathroom #Light).switch_on()\n (#Bathroom #Door).door_open()\n}""}",,/root/llm/JOILang-Server/artifacts/notebook_ga_runs/20260623_180959/baseline_row170_cat6_g3_20260623_181654/candidates/candidates_ga_validation_Qwen_Qwen2.5-Coder-14B-Instruct_gen-3b1ea01c-673b-58d1-52da-f6d4d9796538_508261.csv,[]
4,6,170,gen-442856f6-c8cd-a33f-a37d-5ce08b9fdc88,explicit_preconditions,blocks,NaN,0,20889,94,"{""name"":""BathroomHumidityAndDoorCheck"",""cron"":"""",""period"":0,""code"":""if ((#Bathroom #HumiditySensor).humiditysensor_humidity >= 85 and (#Bathroom #Door).door_doorstate == \""closed\"") {\n (#Bathroom #Light).switch_on()\n (#Bathroom #Door).door_open()\n}""}",,/root/llm/JOILang-Server/artifacts/notebook_ga_runs/20260623_180959/baseline_row170_cat6_g3_20260623_181654/candidates/candidates_ga_validation_Qwen_Qwen2.5-Coder-14B-Instruct_gen-442856f6-c8cd-a33f-a37d-5ce08b9fdc88_503205.csv,[]
5,6,170,gen-72e1069d-e722-5be6-c3bf-6baf915e48f9,compact_json,blocks,NaN,0,12939,90,"{""name"":""BathroomHumidityAndDoorCheck"",""cron"":"""",""period"":0,""code"":""if ((#Bathroom #HumiditySensor).humiditysensor_humidity >= 85 && (#Bathroom #Door).door_doorstate == \""closed\"") { (#Bathroom #Light).switch_on(); (#Bathroom #Door).door_open(); }""}",,/root/llm/JOILang-Server/artifacts/notebook_ga_runs/20260623_180959/baseline_row170_cat6_g3_20260623_181654/candidates/candidates_ga_validation_Qwen_Qwen2.5-Coder-14B-Instruct_gen-72e1069d-e722-5be6-c3bf-6baf915e48f9_507345.csv,[]
6,6,170,gen-72e1069d-e722-5be6-c3bf-6baf915e48f9,compact_json,blocks,NaN,0,12939,90,"{""name"":""BathroomHumidityAndDoorCheck"",""cron"":"""",""period"":0,""code"":""if ((#Bathroom #HumiditySensor).humiditysensor_humidity >= 85

## Cell 5. GT code vs Generated code 병렬 비교

In [18]:
import textwrap
import re

In [19]:
_ = show_gt_vs_generated(
    baseline_run,
    row_no=ROW_NO,
    phase="validation",
    show_all_genomes=True,
    max_items=20,
)

## Cell 6. Feedback / prompt mutation evidence 확인

In [20]:
show_prompt_mutation_evidence(baseline_run)

,best_DETPass,best_avg_DET,best_prompt_tokens,stop_reason
0,100.0,100.0,None,max generations reached


best genome id: gen-442856f6-c8cd-a33f-a37d-5ce08b9fdc88
blocks: ['01', '02', '05', '06']
params: {
  "model": "Qwen/Qwen2.5-Coder-14B-Instruct",
  "temperature": 0.0,
  "few_shot_count": 3,
  "max_tokens": 768,
  "candidate_strategies": [
    "explicit_preconditions",
    "compact_json",
    "minimal",
    "direct",
    "canonical_names_first"
  ]
}
block_params: {
  "02": {
    "few_shot_count": 2,
    "micro_rules": [
      "Use value entries in conditions and function entries in actions.",
      "Prefer canonical_name exactly when available."
    ]
  },
  "05": {
    "repair_mode": "conservative",
    "few_shot_count": 3
  }
}

Diff: /root/llm/JOILang-Server/artifacts/notebook_ga_runs/20260623_180959/baseline_row170_cat6_g3_20260623_181654/ga_block_diffs.jsonl True


,generation,genome_id,base_genome_id,block_id,field,old_value,new_value,mutation_type,mutation_family,compression_level,selected_compression_target,selected_block_id,selected_block_ids,expected_token_delta,feedback_driven,llm_advised,advisor_proposal_id,advisor_batch_id,failure_type_source
0,2,gen-72e1069d-e722-5be6-c3bf-6baf915e48f9,gen-e6f98332-8830-5d32-df5c-e9fedd38fc7e,05,blocks,"[""01"", ""02"", ""03"", ""05"", ""06""]","[""01"", ""02"", ""03"", ""06""]",drop_optional_block,compression,block,05,05,[],-1117,False,False,,,token_overbudget
1,2,gen-6637c9e3-c3e2-716c-624b-f3227734ffac,gen-e6f98332-8830-5d32-df5c-e9fedd38fc7e,genome,blocks,"[""01"", ""02"", ""03"", ""05"", ""06""]","[""01"", ""02"", ""03"", ""06""]",drop_optional_blocks_for_budget,compression,,,,[],0,False,False,,,
2,2,gen-72e1069d-e722-5be6-c3bf-6baf915e48f9,gen-e6f98332-8830-5d32-df5c-e9fedd38fc7e,05,blocks,"[""01"", ""02"", ""03"", ""05"", ""06""]","[""01"", ""02"", ""03"", ""06""]",drop_optional_block,compression,block,05,05,[],-1117,False,False,,,token_overbudget
3,3,gen-016ea579-8fd6-f65f-78f5-12caf55da9da,gen-72e1069d-e722-5be6-c3bf-6baf915e48f9,06,blocks,"[""01"", ""02"", ""03"", ""06""]","[""01"", ""02"", ""03""]",drop_optional_block,compression,block,06,06,[],-53,False,False,,,token_overbudget
4,3,gen-1b17001a-25af-cc4a-c634-cc693b2989e7,gen-72e1069d-e722-5be6-c3bf-6baf915e48f9,genome,params,"{""candidate_strategies"": [""compact_json"", ""explicit_preconditions"", ""direct"", ""minimal""], ""few_shot_count"": 3, ""max_tokens"": 1024, ""model"": ""Qwen/Qwen2.5-Coder-14B-Instruct"", ""temperature"": 0.1}","{""candidate_strategies"": [""compact_json"", ""explicit_preconditions"", ""direct"", ""minimal""], ""few_shot_count"": 3, ""max_tokens"": 1024, ""model"": ""Qwen/Qwen2.5-Coder-14B-Instruct"", ""reasoning_layout"": ""compact_skeleton"", ""temperature"": 0.1}",compact_reasoning_skeleton,compression,,,,[],0,False,False,,,
5,3,gen-1b17001a-25af-cc4a-c634-cc693b2989e7,gen-72e1069d-e722-5be6-c3bf-6baf915e48f9,genome,block_params,"{""02"": {""few_shot_count"": 2, ""micro_rules"": []}, ""05"": {""few_shot_count"": 3, ""repair_mode"": ""conservative""}}","{""02"": {""few_shot_count"": 2, ""micro_rules"": []}, ""05"": {""few_shot_count"": 3, ""repair_mode"": ""conservative""}, ""06"": {}}",compact_reasoning_skeleton,compression,,,,[],0,False,False,,,
6,4,gen-05a1e3d2-44d2-217b-7742-7765a9f6511c,gen-442856f6-c8cd-a33f-a37d-5ce08b9fdc88,05,blocks,"[""01"", ""02"", ""05"", ""06""]","[""01"", ""02"", ""06""]",drop_optional_block,compression,block,05,05,[],-1117,False,False,,,token_overbudget
7,4,gen-c8e3cfea-0b71-4ba0-d6c0-8411673376a3,gen-442856f6-c8cd-a33f-a37d-5ce08b9fdc88,genome,params,"{""candidate_strategies"": [""explicit_preconditions"", ""compact_json"", ""minimal"", ""direct"", ""canonical_names_first""], ""few_shot_count"": 3, ""max_tokens"": 768, ""model"": ""Qwen/Qwen2.5-Coder-14B-Instruct"", ""temperature"": 0.0}","{""candidate_strategies"": [""explicit_preconditions"", ""compact_json""], ""few_shot_count"": 3, ""max_tokens"": 768, ""model"": ""Qwen/Qwen2.5-Coder-14B-Instruct"", ""temperature"": 0.0}",reduce_candidate_strategies,compression,,,,[],0,False,False,,,
8,4,gen-e7a6e2ef-32a3-0300-01fc-5f0c5c785edb,gen-442856f6-c8cd-a33f-a37d-5ce08b9fdc88,06,block_params,"{""02"": {""few_shot_count"": 2, ""micro_rules"": [""Use value entries in conditions and function entries in actions."", ""Prefer canonical_name exactly when available.""]}, ""05"": {""few_shot_count"": 3, ""repair_mode"": ""conservative""}}","{""02"": {""few_shot_count"": 2, ""micro_rules"": [""Use value entries in conditions and function entries in actions."", ""Prefer canonical_name exactly when available.""]}, ""05"": {""few_shot_count"": 3, ""repair_mode"": ""conservative""}, ""06"": {""micro_rules"": [""If code ...",add_targeted_repair_hint,accuracy_repair,,,,[],0,True,False,,,semantic_error


## Cell 7. 실제 prompt 원문 확인

In [21]:
baseline_prompt_log = inspect_prompt_log_from_candidates(
    baseline_run,
    row_no=ROW_NO,
    phase="validation",
    row_index=0,
    max_chars=9000,
)

prompt log: /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/logs/ga_validation_Qwen_Qwen2.5-Coder-14B-Instruct_gen-016ea579-8fd6-f65f-78f5-12caf55da9da_500845/row_170_cand_1.json True

=== SYSTEM HEAD ===
You are a deterministic JOILang generation engine. The natural-language command may be written in English or Korean. If it is Korean, translate it internally to the closest intent-preserving English meaning before reasoning. Follow the user instructions exactly and return only the requested JSON object.

=== USER HEAD ===
Language handling rule:
- The command may be English or Korean.
- If it is Korean, translate it internally to the closest English command intent first.
- Do not output the translation. Output only the final JOI JSON object.

You are a deterministic JOILang generator working against a connected-device capability map.

Global rules:
- Use only the provided service_list_snippet, which is derived from `connected_devices`, retrieval shortlist fallback, and the 

## Cell 8. Manual feedback rule 설계

In [22]:
if int(gt["category"]) == 5:
    MANUAL_RULES = [
        "For temporal/sequential commands, preserve the action order exactly as stated by the user.",
        "Represent explicit delay or after-N-time requirements with the canonical JOILang delay/clock form used in the dataset.",
        "Do not collapse two sequential actions into one action, and do not add unrelated conditions or helper state.",
        "Prefer minimal schema-valid code: device/function arguments must match the service schema exactly.",
    ]
elif int(gt["category"]) == 6:
    MANUAL_RULES = [
        "For compound-condition commands, preserve all conditions and boolean connectors exactly.",
        "Do not execute the action unless the complete condition is satisfied.",
        "Map sensor values only in condition expressions and actuator functions only in action statements.",
        "Prefer minimal schema-valid code with canonical device/function names.",
    ]
else:
    MANUAL_RULES = [
        "Prefer minimal schema-valid JOILang code that directly matches the command intent.",
        "Do not add unrelated actions, variables, conditions, or delays.",
    ]

display(pd.DataFrame({"manual_rule": MANUAL_RULES}))

,manual_rule
0,"For compound-condition commands, preserve all conditions and boolean connectors exactly."
1,Do not execute the action unless the complete condition is satisfied.
2,Map sensor values only in condition expressions and actuator functions only in action statements.
3,Prefer minimal schema-valid code with canonical device/function names.


## Cell 9. Manual feedback을 patched genome/source_file에 강제 반영

In [23]:
patched_genome_path = make_manual_patched_genome(
    baseline_run / "best_genome.json",
    rules=MANUAL_RULES,
    target_block_id="02",
    patch_name=f"row{ROW_NO:03d}_cat{gt['category']}_manual_rules_{ts()}",
    force_source_file_patch=True,
)

print("patched_genome_path:", patched_genome_path)
print(Path(patched_genome_path).read_text(encoding="utf-8")[:5000])

patched_genome_path: /root/llm/JOILang-Server/artifacts/notebook_ga_runs/20260623_180959/patched_genomes/row170_cat6_manual_rules_20260623_182339.json
{
  "id": "gen-442856f6-c8cd-a33f-a37d-5ce08b9fdc88__row170_cat6_manual_rules_20260623_182339",
  "seed": 500353188,
  "blocks": [
    "01",
    "02",
    "05",
    "06"
  ],
  "params": {
    "model": "Qwen/Qwen2.5-Coder-14B-Instruct",
    "temperature": 0.0,
    "few_shot_count": 3,
    "max_tokens": 768,
    "candidate_strategies": [
      "explicit_preconditions",
      "compact_json",
      "minimal",
      "direct",
      "canonical_names_first"
    ]
  },
  "block_params": {
    "02": {
      "few_shot_count": 2,
      "micro_rules": [
        "Use value entries in conditions and function entries in actions.",
        "Prefer canonical_name exactly when available.",
        "For compound-condition commands, preserve all conditions and boolean connectors exactly.",
        "Do not execute the action unless the complete condition is

## Cell 10. Before/after genome diff

In [24]:
_ = diff_json_objects(
    baseline_run / "best_genome.json",
    patched_genome_path,
    max_lines=260,
)

--- /root/llm/JOILang-Server/artifacts/notebook_ga_runs/20260623_180959/baseline_row170_cat6_g3_20260623_181654/best_genome.json
+++ /root/llm/JOILang-Server/artifacts/notebook_ga_runs/20260623_180959/patched_genomes/row170_cat6_manual_rules_20260623_182339.json
@@ -11,13 +11,35 @@
     "prompt_hash": "edf2b30eb0b6627d",
     "rule_signature": "28de14a34f971ccc"
   },
+  "_notebook_manual_block_patch": {
+    "marker": "NOTEBOOK MANUAL FOCUS RULES",
+    "patched_block_path": "/root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/blocks/generated/02_generator_prompt__row170_cat6_manual_rules_20260623_182339.txt",
+    "source_path": "/root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/blocks/02_generator_prompt.txt",
+    "target_block_id": "02"
+  },
+  "_notebook_manual_patch": {
+    "base_genome_path": "/root/llm/JOILang-Server/artifacts/notebook_ga_runs/20260623_180959/baseline_row170_cat6_g3_20260623_181654/best_genome.json",
+    "created_at": "2026-06-23T18:23:39",


## Cell 11. 같은 row 재실행

In [ ]:
patched_run = run_ga(
    label=f"manual_patch_row{ROW_NO:03d}_cat{gt['category']}_g3_{ts()}",
    scope_args=["--start-row", str(ROW_NO), "--end-row", str(ROW_NO)],
    tuning_args=ROW_TUNING,
    genome_json=patched_genome_path,
)

summary_after, best_after = summarize_ga_run(patched_run)

if detect_worker_crash(patched_run):
    raise RuntimeError("worker_crash detected in patched run.")


[CMD]
/root/llm/je/bin/python -u /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/scripts/run_ga_search.py --profile version0_15 --genome-json /root/llm/JOILang-Server/artifacts/notebook_ga_runs/20260623_180959/patched_genomes/row170_cat6_manual_rules_20260623_182339.json --dataset /root/llm/JOILang-Server/datasets/JOICommands-280.csv --service-schema /root/llm/JOILang-Server/datasets/service_list_ver2.0.1.json --model-key qwen25_coder_14b --llm-mode worker --candidate-k 1 --repair-attempts 0 --det-profile strict --selection-mode redesign --fitness-mode phase_aware --mutation-mode cloudless_decompiler --category-balance-mode guard --token-penalty-mode hybrid --stop-controller-mode active --reasoning-mutation-mode auto --intent-hint-mode auto --feedback-guided-mutation --enable-compression-mutation --enable-prompt-decompiler --enable-rendered-prompt-dedupe --enable-pareto-archive --enable-group-specialist-archives --full-run --force --progress verbose --retries 0 --target-det

## Cell 12. Patched run compact table

In [ ]:
patched_candidates = show_tested_rows_only(
    patched_run,
    row_no=ROW_NO,
    phase="validation",
    max_rows=30,
)

## Cell 13. Patched run GT vs Generated

In [ ]:
_ = show_gt_vs_generated(
    patched_run,
    row_no=ROW_NO,
    phase="validation",
    show_all_genomes=True,
    max_items=20,
)

## Cell 14. GT / before / after 3열 병렬 비교

In [ ]:
_ = show_before_after_generated_parallel(
    baseline_run,
    patched_run,
    row_no=ROW_NO,
    phase="validation",
)

## Cell 15. Before/after evaluation table

In [ ]:
comparison = compare_two_runs(
    baseline_run,
    patched_run,
    row_no=ROW_NO,
    phase="validation",
)

## Cell 16. Patched prompt 원문 확인

In [ ]:
patched_prompt_log = inspect_prompt_log_from_candidates(
    patched_run,
    row_no=ROW_NO,
    phase="validation",
    row_index=0,
    max_chars=9000,
)

if patched_prompt_log:
    user_prompt = patched_prompt_log.get("request", {}).get("user", "")
    print("\nManual rule visibility in final rendered prompt:")
    for rule in MANUAL_RULES:
        print(f"- {rule[:80]}... =>", rule in user_prompt)

# Category Test

## Cell 17. Category sweep

In [ ]:
# ============================================================
# Cell 17A. Category-level baseline GA
# ============================================================

RUN_CATEGORY_BASELINE = True

CATEGORY_TO_RUN = 5          # 5 또는 6부터 추천
CATEGORY_LIMIT_PER_CATEGORY = 5

CATEGORY_TUNING = [
    "--population", "6",
    "--gens", "3",
    "--min-generations", "2",
    "--max-generations", "3",
    "--sample-size", "4",
    "--validation-size", "4",
    "--cheap-eval-limit", "2",
    "--plateau-window", "1",
    "--disruptive-max-attempts", "1",
    "--timeout-sec", "3600",
    "--limit-per-category", str(CATEGORY_LIMIT_PER_CATEGORY),
]

category_baseline_run = None

if RUN_CATEGORY_BASELINE:
    category_baseline_run = run_ga(
        label=f"category{CATEGORY_TO_RUN}_baseline_g3_{ts()}",
        scope_args=["--category", str(CATEGORY_TO_RUN)],
        tuning_args=CATEGORY_TUNING,
    )

    summarize_ga_run(category_baseline_run)

    if detect_worker_crash(category_baseline_run):
        raise RuntimeError("worker_crash detected in category baseline run.")

In [ ]:
# ============================================================
# Cell 17B. Show tested rows only
# ============================================================

category_baseline_candidates = show_tested_rows_only(
    category_baseline_run,
    row_no=None,
    phase="validation",
    max_rows=80,
)

In [ ]:
# ============================================================
# Cell 17C. Pick representative rows for prompt inspection
# ============================================================

def list_tested_row_nos(run_dir, phase="validation"):
    df = collect_tested_candidate_rows(run_dir, row_no=None, phase=phase)

    if df.empty or "row_no" not in df.columns:
        return []

    row_nos = (
        df["row_no"]
        .dropna()
        .astype(int)
        .drop_duplicates()
        .tolist()
    )

    return row_nos

CATEGORY_TESTED_ROWS = list_tested_row_nos(
    category_baseline_run,
    phase="validation",
)

print("CATEGORY_TESTED_ROWS:", CATEGORY_TESTED_ROWS)

# 우선 대표 row는 첫 번째 validation row로 선택.
# 필요하면 직접 바꾸면 됨. 예: CATEGORY_DEBUG_ROWS = [137, 141]
CATEGORY_DEBUG_ROWS = CATEGORY_TESTED_ROWS[:3]

print("CATEGORY_DEBUG_ROWS:", CATEGORY_DEBUG_ROWS)

for r in CATEGORY_DEBUG_ROWS:
    gt = extract_gt_fields(r)
    print("\n" + "=" * 100)
    print("ROW:", r, "CATEGORY:", gt["category"])
    print("ENG:", gt["eng"])
    print("KOR:", gt["kor"])

In [ ]:
# 더 강하게 실패 row를 우선 보기 위한 아래 함수도 추가

def pick_suspicious_rows(run_dir, phase="validation", topn=5):
    df = collect_tested_candidate_rows(run_dir, row_no=None, phase=phase)

    if df.empty or "row_no" not in df.columns:
        return []

    df = df.copy()

    # 에러/빈 output 우선
    df["has_error"] = (
        df.get("generation_error_type", "")
        .astype(str)
        .str.len()
        .gt(0)
        if "generation_error_type" in df.columns
        else False
    )

    df["empty_generated"] = (
        df.get("generated_code", "")
        .astype(str)
        .str.strip()
        .eq("")
        if "generated_code" in df.columns
        else False
    )

    # pass/failure 관련 컬럼이 있으면 활용
    score_cols = [
        c for c in df.columns
        if any(k in c.lower() for k in ["det", "score", "pass", "failure"])
    ]

    display_cols = [
        c for c in [
            "row_no",
            "category",
            "genome_id",
            "candidate_strategy",
            "generation_error_type",
            "generated_code",
        ]
        if c in df.columns
    ] + score_cols

    display(df[display_cols].head(80))

    suspicious = df[df["has_error"] | df["empty_generated"]]

    if suspicious.empty:
        suspicious = df

    return (
        suspicious["row_no"]
        .dropna()
        .astype(int)
        .drop_duplicates()
        .head(topn)
        .tolist()
    )

CATEGORY_DEBUG_ROWS = pick_suspicious_rows(
    category_baseline_run,
    phase="validation",
    topn=3,
)

print("CATEGORY_DEBUG_ROWS:", CATEGORY_DEBUG_ROWS)

In [ ]:
# ============================================================
# Cell 17D. GT vs Generated for selected category rows
# ============================================================

for r in CATEGORY_DEBUG_ROWS:
    print("\n" + "=" * 120)
    print(f"GT vs Generated | row={r}")
    _ = show_gt_vs_generated(
        category_baseline_run,
        row_no=r,
        phase="validation",
        show_all_genomes=True,
        max_items=10,
    )

In [ ]:
# ============================================================
# Cell 17E. Inspect actual prompt for representative category row
# ============================================================

CATEGORY_PROMPT_ROW = CATEGORY_DEBUG_ROWS[0]

category_baseline_prompt_log = inspect_prompt_log_from_candidates(
    category_baseline_run,
    row_no=CATEGORY_PROMPT_ROW,
    phase="validation",
    row_index=0,
    max_chars=9000,
)

In [ ]:
# ============================================================
# Cell 17F. Design category-level manual feedback rules
# ============================================================

if CATEGORY_TO_RUN == 5:
    CATEGORY_MANUAL_RULES = [
        "For temporal/sequential commands, preserve the exact order of all requested actions.",
        "If the command says after N seconds/minutes/hours, represent that interval explicitly with the canonical delay or clock form used by the dataset.",
        "Do not merge two sequential actions into a single action.",
        "Do not add unrelated conditions, loops, helper variables, or extra device actions.",
        "For category-5 rows, prioritize action order and delay placement over prompt compression.",
    ]

elif CATEGORY_TO_RUN == 6:
    CATEGORY_MANUAL_RULES = [
        "For compound-condition commands, preserve every condition and every boolean connector exactly.",
        "Use sensor/value services only inside condition expressions.",
        "Use actuator/function services only in action statements.",
        "Do not execute any action unless the full compound condition is satisfied.",
        "For category-6 rows, prioritize condition completeness and canonical service mapping over prompt compression.",
    ]

else:
    CATEGORY_MANUAL_RULES = [
        "Prefer minimal schema-valid JOILang code that directly matches the command.",
        "Do not add unrelated actions, variables, conditions, or delays.",
    ]

display(pd.DataFrame({"category_manual_rule": CATEGORY_MANUAL_RULES}))

In [ ]:
# ============================================================
# Cell 17G. Patch category best genome
# ============================================================

category_patched_genome_path = make_manual_patched_genome(
    category_baseline_run / "best_genome.json",
    rules=CATEGORY_MANUAL_RULES,
    target_block_id="02",
    patch_name=f"category{CATEGORY_TO_RUN}_manual_rules_{ts()}",
    force_source_file_patch=True,
)

print("category_patched_genome_path:", category_patched_genome_path)
print(Path(category_patched_genome_path).read_text(encoding="utf-8")[:5000])

In [ ]:
# ============================================================
# Cell 17H. Category before/after genome diff
# ============================================================

_ = diff_json_objects(
    category_baseline_run / "best_genome.json",
    category_patched_genome_path,
    max_lines=260,
)

In [ ]:
# ============================================================
# Cell 17I. Re-run same category with patched genome
# ============================================================

RUN_CATEGORY_PATCHED = True

category_patched_run = None

if RUN_CATEGORY_PATCHED:
    category_patched_run = run_ga(
        label=f"category{CATEGORY_TO_RUN}_manual_patch_g3_{ts()}",
        scope_args=["--category", str(CATEGORY_TO_RUN)],
        tuning_args=CATEGORY_TUNING,
        genome_json=category_patched_genome_path,
    )

    summarize_ga_run(category_patched_run)

    if detect_worker_crash(category_patched_run):
        raise RuntimeError("worker_crash detected in category patched run.")

In [ ]:
# ============================================================
# Cell 17J. Category before/after evaluation table
# ============================================================

category_comparison = compare_two_runs(
    category_baseline_run,
    category_patched_run,
    row_no=None,
    phase="validation",
)

In [ ]:
# ============================================================
# Cell 17K. GT / Before / After for selected category rows
# ============================================================

for r in CATEGORY_DEBUG_ROWS:
    print("\n" + "=" * 120)
    print(f"Before/After comparison | row={r}")

    _ = show_before_after_generated_parallel(
        category_baseline_run,
        category_patched_run,
        row_no=r,
        phase="validation",
    )

In [ ]:
# ============================================================
# Cell 17L. Verify manual rules in patched category prompt
# ============================================================

category_patched_prompt_log = inspect_prompt_log_from_candidates(
    category_patched_run,
    row_no=CATEGORY_PROMPT_ROW,
    phase="validation",
    row_index=0,
    max_chars=9000,
)

if category_patched_prompt_log:
    user_prompt = category_patched_prompt_log.get("request", {}).get("user", "")

    print("\nCategory manual rule visibility in final rendered prompt:")
    for rule in CATEGORY_MANUAL_RULES:
        print(f"- {rule[:90]}... =>", rule in user_prompt)

# Full Test

## Cell 18. Full 280 rows × 10 generations

In [ ]:
RUN_FULL_10GEN = False

FULL_TUNING = [
    "--population", "16",
    "--gens", "10",
    "--min-generations", "5",
    "--max-generations", "10",
    "--sample-size", "40",
    "--validation-size", "40",
    "--cheap-eval-limit", "20",
    "--plateau-window", "3",
    "--disruptive-max-attempts", "3",
    "--timeout-sec", "7200",
]

full_run = None
if RUN_FULL_10GEN:
    full_run = run_ga(
        label=f"cloudless_full280_g10_{ts()}",
        scope_args=[
            "--category", "1", "--category", "2", "--category", "3", "--category", "4",
            "--category", "5", "--category", "6", "--category", "7", "--category", "8",
        ],
        tuning_args=FULL_TUNING,
    )
    summarize_ga_run(full_run)

## Cell 19. 최종 그래프와 table

In [ ]:
analysis_runs = [baseline_run, patched_run]
if "category_runs" in globals():
    analysis_runs += category_runs
if full_run:
    analysis_runs.append(full_run)

display(collect_run_table(analysis_runs))
_ = plot_generation_history(analysis_runs)

for rd in analysis_runs:
    pareto = read_csv_if_exists(Path(rd) / "pareto_rows.csv")
    if not pareto.empty and {"det_pass_rate", "avg_prompt_tokens"}.issubset(pareto.columns):
        plt.figure(figsize=(6, 4))
        plt.scatter(pareto["avg_prompt_tokens"], pareto["det_pass_rate"])
        plt.xlabel("Avg prompt tokens")
        plt.ylabel("DET pass rate")
        plt.title(f"Pareto scatter: {Path(rd).name}")
        plt.grid(True, alpha=0.3)
        plt.show()